In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
import os
from tqdm.auto import tqdm

# ==========================================
# 1. METRICS & HELPERS
# ==========================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0.0: return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

class LgbTqdmCallback:
    def __init__(self, max_iter, desc="Boosting"):
        self.max_iter = max_iter
        self.pbar = tqdm(total=max_iter, desc=desc, leave=False)
    def __call__(self, env):
        self.pbar.update(1)
        if env.evaluation_result_list:
            score = env.evaluation_result_list[0][2]
            self.pbar.set_postfix({'val_rmse': f'{score:.5f}'})
    def __del__(self):
        self.pbar.close()

# ==========================================
# 2. PREDICTIVE CLUSTERING LOGIC
# ==========================================
def get_predictive_clusters(X, y, codes, n_clusters=5):
    base_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, n_jobs=-1, random_state=42, verbosity=-1)
    base_model.fit(X, y)
    preds = base_model.predict(X)
    
    profile_df = pd.DataFrame({'code': codes, 'actual': y, 'pred': preds, 'error': preds - y})
    
    stock_profiles = profile_df.groupby('code').agg(
        mean_actual=('actual', 'mean'),
        mean_pred=('pred', 'mean'),
        mean_error=('error', 'mean'),
        std_error=('error', 'std')
    ).fillna(0) 
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    stock_profiles['cluster'] = kmeans.fit_predict(stock_profiles)
    
    return stock_profiles['cluster'].to_dict()

# ==========================================
# 3. DATA LOADING & PREP
# ==========================================
print("Loading data...")
base_path = "./" 
try:
    train = pd.read_parquet(os.path.join(base_path, "train.parquet"))
    test = pd.read_parquet(os.path.join(base_path, "test.parquet"))
except FileNotFoundError:
    print("Warning: Paths not found, please update base_path for Kaggle environment.")

cat_cols = ['code', 'sub_code', 'sub_category']
feature_cols = [c for c in train.columns if c.startswith('feature_')]

for col in cat_cols:
    le = LabelEncoder()
    full_data = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(full_data)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8,
    'verbosity': -1, 'seed': 42, 'n_jobs': -1
}
N_ESTIMATORS = 800
N_CLUSTERS = 5 
horizons = train['horizon'].unique()

print("\n" + "="*50)
print(" RUN 1: 90/10 TEMPORAL VALIDATION (UNBIASED)")
print("="*50)

all_val_dfs = []

for h in tqdm(horizons, desc="Horizons (Validation)"):
    train_h = train[train['horizon'] == h].copy()
    
    split_idx = int(train_h['ts_index'].max() * 0.90)
    tr_df = train_h[train_h['ts_index'] <= split_idx].copy()
    val_df = train_h[train_h['ts_index'] > split_idx].copy()
    
    tqdm.write(f"  [Horizon {h}] Clustering stocks based on predictive similarity...")
    cluster_map = get_predictive_clusters(
        X=tr_df[feature_cols], y=tr_df['y_target'], codes=tr_df['code'], n_clusters=N_CLUSTERS
    )
    
    tr_df['cluster'] = tr_df['code'].map(cluster_map).fillna(0).astype(int)
    val_df['cluster'] = val_df['code'].map(cluster_map).fillna(0).astype(int)
    val_df['val_preds'] = 0.0
    
    for c_id in range(N_CLUSTERS):
        c_tr = tr_df[tr_df['cluster'] == c_id]
        c_val = val_df[val_df['cluster'] == c_id]
        
        if len(c_tr) == 0: continue
            
        model = lgb.LGBMRegressor(**params, n_estimators=N_ESTIMATORS)
        tqdm_cb = LgbTqdmCallback(max_iter=N_ESTIMATORS, desc=f"Cluster {c_id} Trees")
        callbacks = [tqdm_cb]
        eval_set = None; eval_weight = None
        
        if len(c_val) > 0:
            eval_set = [(c_val[feature_cols], c_val['y_target'])]
            eval_weight = [c_val['weight']]
            callbacks.append(lgb.early_stopping(50, verbose=False))
        
        model.fit(
            c_tr[feature_cols], c_tr['y_target'],
            sample_weight=c_tr['weight'],
            eval_set=eval_set, eval_sample_weight=eval_weight,
            callbacks=callbacks
        )
        tqdm_cb.pbar.close() 
        
        if len(c_val) > 0:
            val_df.loc[val_df['cluster'] == c_id, 'val_preds'] = model.predict(c_val[feature_cols])
            
    score = weighted_rmse_score(val_df['y_target'].values, val_df['val_preds'].values, val_df['weight'].values)
    tqdm.write(f"✓ Horizon {h} Validation Score: {score:.5f}")
    all_val_dfs.append(val_df)

# Exact Global Scoring for Run 1
full_val_dataset = pd.concat(all_val_dfs, axis=0)
exact_global_val_score = weighted_rmse_score(
    full_val_dataset['y_target'].values, full_val_dataset['val_preds'].values, full_val_dataset['weight'].values
)
print(f"\nExact Global Validation Score (Run 1): {exact_global_val_score:.5f}")


print("\n" + "="*50)
print(" RUN 2: FULL TRAINING PIPELINE (100% DATA)")
print("="*50)

test['y_target'] = 0.0
all_train_dfs = []

for h in tqdm(horizons, desc="Horizons (Full Training)"):
    train_h = train[train['horizon'] == h].copy()
    test_h = test[test['horizon'] == h].copy()
    train_h['train_preds'] = 0.0 # Container for training predictions
    
    tqdm.write(f"  [Horizon {h}] Clustering all stocks...")
    cluster_map = get_predictive_clusters(
        X=train_h[feature_cols], y=train_h['y_target'], codes=train_h['code'], n_clusters=N_CLUSTERS
    )
    
    train_h['cluster'] = train_h['code'].map(cluster_map).fillna(0).astype(int)
    test_h['cluster'] = test_h['code'].map(cluster_map).fillna(0).astype(int)
    
    for c_id in range(N_CLUSTERS):
        c_tr = train_h[train_h['cluster'] == c_id]
        c_te = test_h[test_h['cluster'] == c_id]
        
        if len(c_tr) == 0: continue
            
        n_est_run2 = int(N_ESTIMATORS * 0.8)
        model = lgb.LGBMRegressor(**params, n_estimators=n_est_run2)
        tqdm_cb = LgbTqdmCallback(max_iter=n_est_run2, desc=f"Cluster {c_id} Trees")
        
        model.fit(c_tr[feature_cols], c_tr['y_target'], sample_weight=c_tr['weight'], callbacks=[tqdm_cb])
        tqdm_cb.pbar.close() 
        
        # Predict on the training data for our sanity check
        train_h.loc[c_tr.index, 'train_preds'] = model.predict(c_tr[feature_cols])
        
        # Predict on the actual test set for submission
        if len(c_te) > 0:
            preds = model.predict(c_te[feature_cols])
            test.loc[c_te.index, 'y_target'] = preds
            
    # Score the training set for this horizon
    train_score = weighted_rmse_score(train_h['y_target'].values, train_h['train_preds'].values, train_h['weight'].values)
    tqdm.write(f"⚠ Horizon {h} TRAINING Score: {train_score:.5f} (Warning: Overfit Proxy)")
    all_train_dfs.append(train_h)

# Exact Global Scoring for Run 2 (Training set)
full_train_dataset = pd.concat(all_train_dfs, axis=0)
exact_global_train_score = weighted_rmse_score(
    full_train_dataset['y_target'].values, full_train_dataset['train_preds'].values, full_train_dataset['weight'].values
)
print(f"\nExact Global TRAINING Score (Run 2): {exact_global_train_score:.5f} (Sanity Check Only!)")

# ==========================================
# 4. FINAL SUBMISSION
# ==========================================
submission = test[['id', 'y_target']].copy()
submission.to_csv('submission.csv', index=False)
print("\nFinal submission file 'submission.csv' generated successfully!")

Loading data...

 RUN 1: 90/10 TEMPORAL VALIDATION (UNBIASED)


Horizons (Validation):   0%|          | 0/4 [00:03<?, ?it/s]

  [Horizon 25] Clustering stocks based on predictive similarity...


Horizons (Validation):  25%|██▌       | 1/4 [00:13<00:40, 13.53s/it]

✓ Horizon 25 Validation Score: 0.12522


...

⚠ Horizon 3 TRAINING Score: 0.40378 (Warning: Overfit Proxy)


Horizons (Full Training):  75%|███████▌  | 3/4 [02:34<00:51, 51.75s/it]

  [Horizon 10] Clustering all stocks...


Horizons (Full Training): 100%|██████████| 4/4 [03:31<00:00, 52.82s/it]


⚠ Horizon 10 TRAINING Score: 0.46102 (Warning: Overfit Proxy)

Exact Global TRAINING Score (Run 2): 0.47714 (Sanity Check Only!)

Final submission file 'submission.csv' generated successfully!
